# 🛒 شبیه‌ساز یک‌سالهٔ فروشگاه — سوپری‌من (Google Colab)

این نوت‌بوک **یک سال کامل** از زندگی یک فروشگاه بزرگ را با کدِ خودِ برنامه شبیه‌سازی می‌کند و در پایان یک **فایل پشتیبان واقعی** می‌سازد که هم روی **ویندوز** و هم روی **اندروید** بازیابی می‌شود.

## چه چیزی شبیه‌سازی می‌شود؟
- ✅ ~۱۰۰ فاکتور در روز × ۳۶۵ روز (فروش از مسیر واقعی صندوق برنامه — نه دادهٔ ساختگی)
- ✅ **کل بانک کالاهای پیش‌فرض** (۱۳٬۵۷۰ کالا) با تاریخچهٔ ورود کالا
- ✅ صدها مشتری با عادت‌های واقعی (VIP، معمولی، ازدست‌رفته) + فروش نسیه
- ✅ چک‌های دریافتی/پرداختی، هزینه‌ها (اجاره، قبض‌ها) و **حقوق ماهانهٔ کارکنان**
- ✅ **چند کارتخون (صندوق)** و **شیفت‌بندی هفتگی کارکنان** با جمع‌بندی و مغایرت جزئی
- ✅ **چندین انبارگردانی** در طول سال (شمارش، تأیید و اصلاح از مسیر واقعی برنامه)
- ✅ اجرای پیشنهادهای **هوش فروشگاه** در طول سال و سنجش نتیجهٔ واقعی‌شان
- ✅ گزارش‌ها، حسابداری دوطرفه، سود و زیان — همه از موتور خود برنامه

## خروجی چیست؟
یک فایل `supermarket_sim_365d_*.db.gz` — درست با همان قالبی که خود برنامه پشتیبان می‌گیرد:
- **ویندوز:** تنظیمات ← پشتیبان‌گیری ← «بازیابی از فایل…»
- **اندروید:** تنظیمات ← پشتیبان ← انتخاب همین فایل (اپ موبایل، پشتیبان ویندوز را بومی ایمپورت می‌کند)

## دربارهٔ GPU (روراست)
این شبیه‌سازی **موتور زبانی ندارد** (نسخهٔ ۴.۶ به بعد)؛ همه‌چیز محاسبات پایگاه‌داده و حسابداری است و **به GPU نیازی نیست**. چیزی که کولب می‌دهد، CPU سریع و دیسک پرسرعت است — همین باعث می‌شود سالِ کامل در حدود چند دقیقه تا نیم‌ساعت (بسته به ترافیک کولب) ساخته شود. اگر ران‌تایم GPU گرفتید هم مشکلی نیست؛ فقط استفاده نمی‌شود.

---
🔴 **قبل از شروع:** از منوی `Runtime` ← `Run all` را بزنید و صبر کنید. مراحل به‌ترتیب خودشان اجرا می‌شوند و نوار پیشرفت فارسی نشان داده می‌شود.

In [ ]:
# ── ۱/۵ ───────────────── تنظیمات (این خانه را مطابق میل تغییر دهید) ─────────────────
DAYS       = 365      # طول شبیه‌سازی به روز
PER_DAY    = 100      # میانگین فاکتور در روز (فروشگاه بزرگ)
SEED       = 1404     # بذر تصادفی — همان بذر = همان فروشگاه (تکرارپذیر)
BRANCH     = "arena/01a0ca3c-super-system"   # شاخهٔ رسمی پروژه (می‌توانید نسخه/تگ بگذارید)
REPO_URL   = "https://github.com/khajavy8056/Super-system-.git"
RESUMABLE  = True     # اگر کولب قطع شد، دفعهٔ بعد از همان روز ادامه می‌دهد

from pathlib import Path
OUT = Path("/content") / f"supermarket_sim_{DAYS}d_{SEED}.db.gz"
WORK = Path("/content/sim_work")          # پوشهٔ ادامهٔ کار
print(f"تنظیم شد: {DAYS} روز · ~{PER_DAY} فاکتور/روز · بذر {SEED}")
print(f"خروجی: {OUT}")

In [ ]:
# ── ۲/۵ ───────────────── دانلود پروژهٔ خودمان از گیت‌هاب ─────────────────
import subprocess, os
REPO = "/content/Super-system-"
if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
r = subprocess.run(["git", "-C", REPO, "checkout", BRANCH], capture_output=True, text=True)
if r.returncode != 0:
    print(f"شاخهٔ {BRANCH} در دسترس نبود؛ از شاخهٔ پیش‌فرض استفاده می‌شود. ({r.stderr.strip()[:120]})")
head = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
ver = ""
initpy = Path(REPO) / "backend/app/__init__.py"
if initpy.exists():
    ver = [l.split('=')[1].strip().strip('\"') for l in initpy.read_text().splitlines() if l.startswith("__version__")][0]
print(f"✓ پروژه آماده شد — نسخهٔ {ver} (کامیت {head})")

In [ ]:
# ── ۳/۵ ───────────────── نصب نیازمندی‌ها (حدود ۱ دقیقه) ─────────────────
!pip install -q -r /content/Super-system-/backend/requirements.txt tqdm
print("✓ نصب کامل شد")

## ۴/۵ — اجرای شبیه‌سازیِ یک سال

نوار پیشرفت پایین، پیشرفت واقعی ساخت است (روز به روز).
اگر کولب وسط کار قطع شد، همین خانه را دوباره اجرا کنید — از همان روز ادامه می‌یابد (چون `RESUMABLE` روشن است).

In [ ]:
# ── ۴/۵ ───────────────── شبیه‌سازی کامل یک سال ─────────────────
import sys
sys.argv = ["simulate_year.py", "--days", str(DAYS), "--per-day", str(PER_DAY),
            "--seed", str(SEED), "--out", str(OUT)]
if RESUMABLE:
    sys.argv += ["--resume-dir", str(WORK)]
sys.path.insert(0, "/content/Super-system-/tools")
import simulate_year
rc = simulate_year.main(sys.argv[1:])
if rc != 0:
    raise SystemExit(rc)
print("✓ فایل پشتیبان ساخته و اعتبارسنجی شد")

## ۵/۵ — دانلود فایل پشتیبان

فایل زیر را نگه دارید و در برنامه (ویندوز یا اندروید) بازیابی کنید:

| پلتفرم | مسیر بازیابی |
|---|---|
| 💻 ویندوز | تنظیمات ← پشتیبان‌گیری ← «بازیابی از فایل…» |
| 📱 اندروید | تنظیمات ← پشتیبان ← انتخاب فایل (ایمپورت نسخهٔ ویندوز) |

⚠️ بازیابی، داده‌های فعلی همان دستگاه را جایگزین می‌کند — قبل از آن، از دادهٔ واقعی خود پشتیبان بگیرید.

In [ ]:
# ── ۵/۵ ───────────────── دانلود خروجی ─────────────────
import os
if os.path.exists(OUT):
    print(f"حجم فایل: {os.path.getsize(OUT) / 1048576:.1f} مگابایت")
    from google.colab import files
    files.download(str(OUT))
else:
    print("فایل پیدا نشد — خانهٔ ۴ را دوباره اجرا کنید")